# Módulo 02 · Indexación avanzada y `MultiIndex`

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Dominar `.loc`, `.iloc`, `.at` y máscaras booleanas sin ambigüedades.
2. Usar el **índice como llave**: `set_index`, unicidad, ordenamiento y búsquedas rápidas.
3. Entender la **alineación automática por índice** (la fuente de muchos `NaN` misteriosos).
4. Crear y navegar un **`MultiIndex`**: selección parcial, `pd.IndexSlice`, `.xs`.
5. Manipular niveles: `swaplevel`, `droplevel`, `rename_axis`, agregar por nivel.
6. Trabajar con **`MultiIndex` en columnas** y aplanarlas.
7. Usar un **`DatetimeIndex`** con selección por strings parciales.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | Repaso de selección: `loc`, `iloc`, `at`, máscaras |
| 2 | El índice como llave |
| 3 | Alineación por índice |
| 4 | Crear un `MultiIndex` |
| 5 | Seleccionar en un `MultiIndex` |
| 6 | Manipular niveles |
| 7 | `MultiIndex` en columnas |
| 8 | `DatetimeIndex` |
| ✍️ | Ejercicios · ✅ Soluciones |

## 0 · Setup y carga

Cargamos sólo lo que usaremos, declarando los dtypes ahí mismo (repaso del Módulo 01).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

In [ ]:
orders = pd.read_csv(
  DATA / "olist_orders_dataset.csv",
  dtype={"order_status": "category"},
  parse_dates=["order_purchase_timestamp", "order_approved_at",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
  usecols=lambda c: c != "order_delivered_carrier_date",
)

items = pd.read_csv(
  DATA / "olist_order_items_dataset.csv",
  dtype={"order_item_id": "int8"},
  parse_dates=["shipping_limit_date"],
)

products = pd.read_csv(
  DATA / "olist_products_dataset.csv",
  usecols=["product_id", "product_category_name", "product_weight_g"],
  dtype={"product_category_name": "category"},
)

sellers = pd.read_csv(
  DATA / "olist_sellers_dataset.csv",
  dtype={"seller_zip_code_prefix": "str", "seller_state": "category"},
)

{n: df.shape for n, df in [("orders", orders), ("items", items),
                           ("products", products), ("sellers", sellers)]}

## 1 · Repaso de selección: `loc`, `iloc`, `at`, máscaras

| Accesor | Selecciona por | Rebanada `a:b` |
|---|---|---|
| `.loc[filas, cols]` | **etiquetas** (o máscara booleana) | incluye `b` |
| `.iloc[filas, cols]` | **posiciones** enteras | excluye `b` |
| `.at[fila, col]` / `.iat[i, j]` | un solo valor (más rápido) | — |
| `df[col]` / `df[[cols]]` | columnas | — |
| `df[mascara]` | filas por máscara | — |

La diferencia en rebanadas es la fuente de muchos *off-by-one*:

In [ ]:
demo = items.head(6)
print("loc[0:3]  ->", demo.loc[0:3].index.tolist())    # etiquetas 0..3 incluidas
print("iloc[0:3] ->", demo.iloc[0:3].index.tolist())   # posiciones 0..2

In [ ]:
# Filas y columnas a la vez: etiquetas en .loc
items.loc[items["price"] > 500, ["order_id", "price", "freight_value"]]

### Construir máscaras legibles

Combina condiciones con `&`, `|`, `~` (¡siempre con paréntesis!). Métodos útiles: `isin`, `between`, `str.startswith`, `isna`.

In [ ]:
caro = items["price"].between(200, 500)               # incluye ambos extremos por defecto
flete_alto = items["freight_value"] > 30
vendedores_top = items["seller_id"].isin(sellers.loc[sellers["seller_state"] == "PR", "seller_id"])

mascara = caro & (flete_alto | vendedores_top)
print("Filas que cumplen:", mascara.sum())
items.loc[mascara, ["price", "freight_value", "seller_id"]].head()

`.at` es para leer/escribir **un** valor. Es la forma correcta (y rápida) cuando lo necesitas en un bucle:

In [ ]:
print(items.at[0, "price"], items.iat[0, 5])

## 2 · El índice como llave

El `RangeIndex` (0, 1, 2…) no dice nada de tus datos. Poner una columna con significado como índice te da:

- Búsquedas por etiqueta con `.loc[llave]` (con índice único y ordenado son muy rápidas).
- Alineación automática al operar entre objetos (sección 3).
- Joins por índice (`join`, `map`).

In [ ]:
orders_idx = orders.set_index("order_id")
assert orders_idx.index.is_unique, "order_id debería identificar cada pedido"
print("¿Único?   ", orders_idx.index.is_unique)
print("¿Ordenado?", orders_idx.index.is_monotonic_increasing)
orders_idx.head(3)

In [ ]:
un_pedido = orders_idx.index[10]
orders_idx.loc[un_pedido]                 # un Series (una fila)

In [ ]:
orders_idx.loc[[un_pedido]]               # lista de etiquetas -> DataFrame

> ⚠️ **pandas 3.1:** el parámetro `set_index(..., verify_integrity=True)` está **deprecado**. El reemplazo recomendado es comprobar explícitamente `df.index.is_unique`, como en la celda anterior.

### Índices **no únicos**

`order_id` en `items` se repite (un pedido con varios ítems). `.loc` entonces devuelve **varias filas**, y el tipo del resultado depende de cuántas coincidan. Esto rompe código que asume siempre una fila:

In [ ]:
items_by_order = items.set_index("order_id")
print("¿Único?", items_by_order.index.is_unique)

pedido_1_item = items_by_order.index[0]
pedido_6_items = items_by_order.index.value_counts().idxmax()

print(type(items_by_order.loc[pedido_1_item, "price"]).__name__)   # escalar
print(type(items_by_order.loc[pedido_6_items, "price"]).__name__)  # Series

> 💡 Si necesitas siempre un DataFrame, usa una **lista**: `df.loc[[llave]]`.

### Ordenar el índice

Con un índice ordenado pandas usa búsqueda binaria en lugar de recorrer todo. En tablas grandes la diferencia es notable, y para algunas operaciones de `MultiIndex` es **obligatorio**.

In [ ]:
items_by_order = items_by_order.sort_index()
items_by_order.index.is_monotonic_increasing

`reset_index` devuelve el índice a columna. Con `drop=True` lo descarta.

In [ ]:
items_by_order.reset_index().head(2)

## 3 · Alineación por índice

**Toda operación entre dos Series/DataFrames se alinea por etiquetas, no por posición.** Las etiquetas que faltan en alguno producen `NaN`.

Ejemplo: peso de cada producto vendido. `products` tiene el peso por `product_id`; `items` tiene ventas.

In [ ]:
peso = products.set_index("product_id")["product_weight_g"]
peso.head(3)

In [ ]:
# ❌ Sin índice común: se alinea por 0,1,2... ¡asigna el peso de OTRO producto!
mal = items["price"] / products["product_weight_g"]
print("Longitud:", len(mal), "| NaN:", mal.isna().sum())

El cálculo anterior "funcionó" sin error y es **incorrecto**: la fila 0 de `items` se dividió entre la fila 0 de `products`, que es otro producto. Además, como las longitudes difieren, aparecen `NaN` en el sobrante.

✅ La forma correcta es **traer** el valor usando la llave: `Series.map` con una Series indexada actúa como un *lookup*.

In [ ]:
items_con_peso = items.assign(peso_g=items["product_id"].map(peso))
items_con_peso[["product_id", "price", "peso_g"]].head()

### Alineación como herramienta

Cuando **ambos** lados están indexados por la misma llave, la alineación hace exactamente lo que quieres, aunque el orden y el tamaño difieran.

In [ ]:
ventas_por_vendedor = items.groupby("seller_id")["price"].sum()
flete_por_vendedor = items.groupby("seller_id")["freight_value"].sum()

# Desordenamos uno a propósito: el resultado sigue siendo correcto
ratio = flete_por_vendedor.sample(frac=1, random_state=0) / ventas_por_vendedor
ratio.sort_values(ascending=False).head()

### `reindex`: conformar a un índice dado

`reindex` reordena según una lista de etiquetas e inserta `NaN` (o `fill_value`) en las que no existen. Útil para garantizar que aparezcan todas las categorías o todas las fechas.

In [ ]:
estados_brasil = ["SP", "RJ", "MG", "PR", "SC", "RS", "BA", "AM"]
vendedores_por_estado = sellers["seller_state"].value_counts()

vendedores_por_estado.reindex(estados_brasil, fill_value=0)

## 4 · Crear un `MultiIndex`

Un `MultiIndex` es un índice con **varios niveles**: cada etiqueta es una tupla. Aparece naturalmente cuando:

- La llave natural es compuesta: un ítem se identifica por `(order_id, order_item_id)`.
- Agrupas por varias columnas.
- Haces `stack`/`unstack`/`pivot` (Módulo 04).

In [ ]:
items_mi = items.set_index(["order_id", "order_item_id"]).sort_index()
assert items_mi.index.is_unique
items_mi.head(8)

In [ ]:
idx = items_mi.index
print("Niveles   :", idx.nlevels)
print("Nombres   :", idx.names)
print("Únicos por nivel:", [len(l) for l in idx.levels])
print("Primera etiqueta:", idx[0])

Desde un `groupby` con varias llaves:

In [ ]:
items_cat = items.assign(
  categoria=items["product_id"].map(products.set_index("product_id")["product_category_name"]).astype("str"),
  estado_vendedor=items["seller_id"].map(sellers.set_index("seller_id")["seller_state"]).astype("str"),
)

ventas = (
  items_cat.groupby(["estado_vendedor", "categoria"], observed=True)["price"]
  .sum()
  .sort_index()
)
ventas.head(10)

> ⚠️ **¿Por qué el `.astype("str")`?** Si los niveles fueran `category`, las rebanadas por rango (`"a":"c"`) seguirían el **orden de las categorías**, no el alfabético, y fallarían con etiquetas que no son categorías existentes. Para índices que vas a rebanar por texto, conviene usar `str`.

Y construido a mano (útil para plantillas o para `reindex`):

In [ ]:
pd.MultiIndex.from_product(
  [["SP", "RJ"], ["2018-01", "2018-02", "2018-03"]],
  names=["estado", "mes"],
)

## 5 · Seleccionar en un `MultiIndex`

Usaremos la Series `ventas` (índice `estado_vendedor` × `categoria`).

| Quiero… | Sintaxis |
|---|---|
| Un elemento exacto | `s.loc[("SP", "telefonia")]` |
| Todo un valor del 1er nivel | `s.loc["SP"]` (selección parcial) |
| Varios valores del 1er nivel | `s.loc[["SP", "PR"]]` |
| Filtrar en un nivel interior | `s.loc[pd.IndexSlice[:, "telefonia"]]` o `s.xs("telefonia", level=1)` |
| Rangos en varios niveles | `s.loc[pd.IndexSlice["MG":"PR", "a":"c"]]` |
| Máscara por nivel | `s[s.index.get_level_values("categoria").str.startswith("moveis")]` |

In [ ]:
ventas.loc[("SP", "telefonia")]

In [ ]:
ventas.loc["PR"].sort_values(ascending=False).head()    # el nivel 0 desaparece

In [ ]:
ventas.loc[["RJ", "SC"]]                                 # el nivel 0 se conserva

### `pd.IndexSlice`

Para rebanar niveles interiores con `.loc` necesitas `slice(None)` (equivalente a `:`). `pd.IndexSlice` permite escribirlo con la sintaxis natural de rebanadas:

In [ ]:
idx = pd.IndexSlice
ventas.loc[idx[:, "telefonia"]]

In [ ]:
# Rangos en ambos niveles. Requiere índice ORDENADO (por eso hicimos sort_index)
ventas.loc[idx["MG":"PR", "a":"c"]]

> ⚠️ Si el `MultiIndex` no está ordenado, las rebanadas por rango lanzan `UnsortedIndexError` o son lentas. Cuando dudes: `df = df.sort_index()`.

### `.xs` (*cross-section*)

`xs` selecciona un valor en **cualquier nivel** y, por defecto, elimina ese nivel. `drop_level=False` lo conserva.

In [ ]:
ventas.xs("cama_mesa_banho", level="categoria")

In [ ]:
ventas.xs("cama_mesa_banho", level="categoria", drop_level=False).head(3)

### Máscaras sobre un nivel con `get_level_values`

In [ ]:
cat = ventas.index.get_level_values("categoria")
ventas[cat.str.startswith("moveis")]

En un DataFrame con `MultiIndex` todo lo anterior funciona igual, con columnas como segundo argumento de `.loc`:

In [ ]:
pedido = items_mi.index.get_level_values("order_id").value_counts().idxmax()
items_mi.loc[pedido, ["price", "freight_value"]]

In [ ]:
items_mi.loc[(pedido, 2), "price"]          # ítem 2 de ese pedido

## 6 · Manipular niveles

| Método | Hace |
|---|---|
| `swaplevel(i, j)` | intercambia dos niveles |
| `reorder_levels([...])` | reordena todos |
| `droplevel(n)` | elimina un nivel |
| `rename_axis([...])` / `index.set_names` | renombra niveles |
| `groupby(level=...)` | agrega por nivel(es) |
| `sort_index(level=...)` | ordena por nivel |

In [ ]:
por_categoria = ventas.swaplevel().sort_index()
por_categoria.head()

In [ ]:
ventas.rename_axis(["estado", "cat"]).head(3)

In [ ]:
# Total por estado: agregar sobre un nivel
ventas.groupby(level="estado_vendedor", observed=True).sum().sort_values(ascending=False).head()

Un patrón muy común: **participación dentro del grupo**. Dividimos cada valor entre el total de su nivel superior con `transform` (lo veremos a fondo en el Módulo 03):

In [ ]:
participacion = ventas / ventas.groupby(level="estado_vendedor", observed=True).transform("sum")
participacion.loc["SP"].sort_values(ascending=False).head().round(3)

Y cuando quieres "abrir" un nivel a columnas, `unstack` (adelanto del Módulo 04):

In [ ]:
ventas.loc[["SP", "PR", "MG"]].unstack("estado_vendedor", fill_value=0).head()

## 7 · `MultiIndex` en columnas

Aparece al agregar varias columnas con varias funciones:

In [ ]:
resumen = items_cat.groupby("estado_vendedor", observed=True)[["price", "freight_value"]].agg(["sum", "mean", "count"])
resumen.head()

In [ ]:
resumen.columns

In [ ]:
resumen.loc[:, ("price", "mean")].head(3)           # una columna exacta

In [ ]:
resumen.loc[:, idx[:, "mean"]].head(3)               # todas las "mean"

### Aplanar columnas

Para exportar o seguir trabajando, suele convenir aplanar a nombres simples:

In [ ]:
plano = resumen.copy()
plano.columns = ["_".join(col) for col in plano.columns]
plano.head(3)

> 💡 En el Módulo 03 veremos las **agregaciones con nombre** (`agg(precio_total=("price", "sum"))`), que evitan el `MultiIndex` de columnas desde el principio.

## 8 · `DatetimeIndex`

Con fechas en el índice (y **ordenadas**) puedes seleccionar con strings parciales: un año, un mes, un día o un rango.

In [ ]:
orders_t = orders.set_index("order_purchase_timestamp").sort_index()
orders_t.index

In [ ]:
print("2018      :", len(orders_t.loc["2018"]))
print("2018-03   :", len(orders_t.loc["2018-03"]))
print("Q1 2018   :", len(orders_t.loc["2018-01":"2018-03"]))     # ambos extremos incluidos

In [ ]:
orders_t.loc["2018-03-01":"2018-03-05", ["order_status", "order_id"]]

Los atributos del índice (`.year`, `.month`, `.dayofweek`, `.hour`…) sirven para máscaras sin crear columnas:

In [ ]:
nocturnos = orders_t[orders_t.index.hour < 6]
print("Pedidos de madrugada:", len(nocturnos))

> En el Módulo 06 llevaremos esto a `resample`, ventanas móviles y zonas horarias.

---
# ✍️ Ejercicios

Variables disponibles: `orders`, `items`, `products`, `sellers`, `items_cat`, `idx` (= `pd.IndexSlice`).

### Ejercicio 1 · Llave compuesta ⭐

1. Crea `im`: `items` indexado por `(order_id, order_item_id)`, ordenado, y verifica con un `assert` que la llave sea única.
2. `n_multi`: número de pedidos con **más de un** ítem. Usa el índice (pista: `get_level_values` o `groupby(level=...)`).
3. `segundo_item`: el `price` del ítem `2` del pedido `"f97b2bdfbaca9c2f04722d8cbc8cc8a5"`, usando **un solo** `.loc`.
4. `total_pedido`: Series con la suma de `price + freight_value` por `order_id`, calculada agregando sobre el nivel `order_id` de `im`.

In [ ]:
# TU CÓDIGO AQUÍ
im = ...
n_multi = ...
segundo_item = ...
total_pedido = ...

In [ ]:
# Comprobación
if pendiente(im, n_multi, segundo_item, total_pedido):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert isinstance(im.index, pd.MultiIndex) and im.index.names == ["order_id", "order_item_id"]
  assert im.index.is_monotonic_increasing, "Ordena el índice"
  assert n_multi == 27, f"n_multi = {n_multi}"
  assert np.isclose(segundo_item, 43.0), f"segundo_item = {segundo_item}"
  assert total_pedido.index.name == "order_id" and len(total_pedido) == items["order_id"].nunique()
  assert np.isclose(total_pedido.sum(), (items["price"] + items["freight_value"]).sum())
  print("✅ Ejercicio 1 correcto")

### Ejercicio 2 · Lookups con `map` en lugar de merge ⭐

Sin usar `merge` ni `join`:

1. Crea `it` a partir de `items` con dos columnas nuevas: `estado_cliente` (estado del **cliente** que hizo el pedido) y `estado_vendedor`.
   Pistas: el cliente está en `orders` vía `customer_id`, y su estado está en `olist_customers_dataset.csv` (cárgalo tú con el dtype que creas conveniente). Necesitarás encadenar dos `map`.
2. `pct_mismo_estado`: porcentaje (0–100) de ítems donde el vendedor y el cliente están en el mismo estado.

In [ ]:
# TU CÓDIGO AQUÍ
it = ...
pct_mismo_estado = ...

In [ ]:
# Comprobación
if pendiente(it, pct_mismo_estado):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert {"estado_cliente", "estado_vendedor"} <= set(it.columns)
  assert len(it) == len(items), "No debes cambiar el número de filas"
  assert it["estado_cliente"].notna().all() and it["estado_vendedor"].notna().all()
  assert np.isclose(pct_mismo_estado, 33.9286, atol=0.01), f"pct_mismo_estado = {pct_mismo_estado}"
  print("✅ Ejercicio 2 correcto")

### Ejercicio 3 · Navegando un `MultiIndex` ⭐⭐

Usa la Series `ventas` (índice `estado_vendedor` × `categoria`) de la sección 4.

1. `ventas_sp_pr`: ventas de SP y PR, sólo en categorías alfabéticamente entre `"b"` y `"e"` (rebanada `"b":"e"`). Usa `pd.IndexSlice`.
2. `cama_por_estado`: ventas de `"cama_mesa_banho"` por estado (índice = estado), con `.xs`.
3. `top_por_estado`: Series indexada por estado con el **nombre de la categoría** que más vende en cada estado.
   Pista: `groupby(level=...)` + `idxmax` devuelve tuplas; extrae el segundo elemento.

In [ ]:
# TU CÓDIGO AQUÍ
ventas_sp_pr = ...
cama_por_estado = ...
top_por_estado = ...

In [ ]:
# Comprobación
if pendiente(ventas_sp_pr, cama_por_estado, top_por_estado):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert set(ventas_sp_pr.index.get_level_values(0)) <= {"SP", "PR"}
  cats = ventas_sp_pr.index.get_level_values(1)
  assert ((cats >= "b") & (cats <= "e")).all()
  assert len(ventas_sp_pr) == 13, f"len = {len(ventas_sp_pr)}"
  assert cama_por_estado.index.name == "estado_vendedor"
  assert np.isclose(cama_por_estado.sum(), 2635.26)
  assert top_por_estado.loc["SP"] == "cama_mesa_banho", top_por_estado.loc["SP"]
  assert len(top_por_estado) == ventas.index.get_level_values(0).nunique()
  print("✅ Ejercicio 3 correcto")

### Ejercicio 4 · `DatetimeIndex` ⭐⭐

Con `orders_t` (pedidos indexados por fecha de compra):

1. `n_bf`: pedidos hechos en la semana de Black Friday 2017, del **2017-11-20 al 2017-11-26** inclusive.
2. `por_trimestre`: Series con el número de pedidos de **cada trimestre de 2018** presente en los datos, con índice `"2018Q1"`, `"2018Q2"`, … como strings.
   Pista: `orders_t.index.to_period("Q")`.
3. `pct_fin_semana`: porcentaje (0–100) de pedidos hechos en sábado o domingo.

In [ ]:
# TU CÓDIGO AQUÍ
n_bf = ...
por_trimestre = ...
pct_fin_semana = ...

In [ ]:
# Comprobación
if pendiente(n_bf, por_trimestre, pct_fin_semana):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert n_bf == 7, f"n_bf = {n_bf}"
  assert list(por_trimestre.index) == ["2018Q1", "2018Q2", "2018Q3"], list(por_trimestre.index)
  assert por_trimestre.sum() == len(orders_t.loc["2018"])
  assert np.isclose(pct_fin_semana, 22.0, atol=0.01), f"pct_fin_semana = {pct_fin_semana}"
  print("✅ Ejercicio 4 correcto")

### Ejercicio 5 · Reto: columnas jerárquicas ⭐⭐⭐

1. A partir de `items_cat`, crea `res` agrupando por `estado_vendedor` y agregando `price` y `freight_value` con `["sum", "mean"]`.
2. Aplana las columnas a `price_sum`, `price_mean`, `freight_value_sum`, `freight_value_mean`.
3. Agrega la columna `flete_relativo = freight_value_mean / price_mean`.
4. `estados_flete_caro`: lista **ordenada alfabéticamente** de estados (como strings) cuyo `flete_relativo` sea mayor a `0.25`.
5. **Bonus:** recupera el `MultiIndex` de columnas a partir de los nombres planos con `str.rsplit("_", n=1, expand=True)` y guárdalo en `res_mi`.

In [ ]:
# TU CÓDIGO AQUÍ
res = ...
estados_flete_caro = ...
res_mi = ...

In [ ]:
# Comprobación
if pendiente(res, estados_flete_caro, res_mi):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert {"price_sum", "price_mean", "freight_value_sum", "freight_value_mean", "flete_relativo"} <= set(res.columns)
  assert not isinstance(res.columns, pd.MultiIndex)
  assert estados_flete_caro == ["ES", "MT", "SC"], estados_flete_caro
  assert isinstance(res_mi.columns, pd.MultiIndex)
  assert ("freight_value", "mean") in res_mi.columns
  print("✅ Ejercicio 5 correcto")

---
# ✅ Soluciones

### Solución 1

In [ ]:
im = items.set_index(["order_id", "order_item_id"]).sort_index()
assert im.index.is_unique

items_por_pedido = im.groupby(level="order_id").size()
n_multi = int((items_por_pedido > 1).sum())

segundo_item = im.loc[("f97b2bdfbaca9c2f04722d8cbc8cc8a5", 2), "price"]

total_pedido = (im["price"] + im["freight_value"]).groupby(level="order_id").sum()
n_multi, segundo_item, total_pedido.head(3)

*Notas:* el `assert im.index.is_unique` es una aserción gratuita sobre tu modelo de datos: si algún día llegan duplicados, te enteras de inmediato. Como la llave `(order_id, order_item_id)` es única, `.loc[(pedido, 2), "price"]` devuelve un escalar. Alternativa para `n_multi`: `im.index.get_level_values(0).value_counts().gt(1).sum()`.

### Solución 2

In [ ]:
customers = pd.read_csv(
  DATA / "olist_customers_dataset.csv",
  usecols=["customer_id", "customer_state"],
  dtype={"customer_state": "category"},
)

cliente_de_pedido = orders.set_index("order_id")["customer_id"]
estado_de_cliente = customers.set_index("customer_id")["customer_state"]
estado_de_vendedor = sellers.set_index("seller_id")["seller_state"]

it = items.assign(
  estado_cliente=items["order_id"].map(cliente_de_pedido).map(estado_de_cliente),
  estado_vendedor=items["seller_id"].map(estado_de_vendedor),
)

pct_mismo_estado = (
  it["estado_cliente"].astype("str") == it["estado_vendedor"].astype("str")
).mean() * 100
round(pct_mismo_estado, 2)

*Notas:* `map` con una Series indexada es un *lookup*: devuelve un resultado con el **mismo largo** que el original (a diferencia de un `merge`, que puede multiplicar filas si la llave se repite). Convertimos a `str` antes de comparar porque dos columnas `category` con **distintas** listas de categorías no se pueden comparar directamente.

### Solución 3

In [ ]:
ventas_sp_pr = ventas.loc[idx[["PR", "SP"], "b":"e"]]

cama_por_estado = ventas.xs("cama_mesa_banho", level="categoria")

top_por_estado = (
  ventas.groupby(level="estado_vendedor", observed=True)
  .idxmax()
  .str[1]                      # cada valor es una tupla (estado, categoria)
)
top_por_estado.head()

*Notas:* la rebanada `"b":"e"` incluye la etiqueta `"e"` pero **no** `"eletronicos"`, porque en orden de texto `"eletronicos"` > `"e"`. Si quisieras todo lo que empieza con b, c, d o e, el límite sería `"b":"f"` (excluyendo luego `"f"`) o una máscara con `str[0].isin([...])`. `.str[1]` funciona sobre una Series de tuplas para extraer un elemento por posición. Una alternativa más explícita: `.map(lambda t: t[1])`.

### Solución 4

In [ ]:
n_bf = len(orders_t.loc["2017-11-20":"2017-11-26"])

trim = orders_t.loc["2018"].index.to_period("Q")
por_trimestre = trim.value_counts().sort_index()
por_trimestre.index = por_trimestre.index.astype(str)

pct_fin_semana = (orders_t.index.day_of_week >= 5).mean() * 100
n_bf, por_trimestre, round(pct_fin_semana, 2)

*Nota:* con un string de sólo fecha (`"2017-11-26"`) como límite superior, pandas incluye **todo ese día** hasta las 23:59:59. Por eso no hace falta escribir `"2017-11-27"`. `day_of_week` va de lunes = 0 a domingo = 6.

### Solución 5

In [ ]:
res = items_cat.groupby("estado_vendedor", observed=True)[["price", "freight_value"]].agg(["sum", "mean"])
res.columns = ["_".join(c) for c in res.columns]
res = res.assign(flete_relativo=lambda d: d["freight_value_mean"] / d["price_mean"])

estados_flete_caro = sorted(res.index[res["flete_relativo"] > 0.25].astype(str))

# Bonus: de nombres planos a MultiIndex
res_mi = res.drop(columns="flete_relativo")
res_mi.columns = res_mi.columns.str.rsplit("_", n=1, expand=True)
estados_flete_caro, res_mi.head(3)

*Nota del bonus:* usamos `rsplit` con `n=1` (desde la derecha, una vez) porque `freight_value` contiene un `_`. Con `split` normal obtendríamos `("freight", "value", "mean")`.

---
## Resumen del módulo

- `.loc` incluye el extremo; `.iloc` no. `.at`/`.iat` para valores sueltos.
- Indexa por la llave natural con `set_index(..., verify_integrity=True)` y **ordena** (`sort_index`).
- Las operaciones se **alinean por etiquetas**; usa `map` con Series indexadas para *lookups* seguros.
- `MultiIndex`: `.loc[(a, b)]`, selección parcial, `pd.IndexSlice`, `.xs(level=...)`, `get_level_values`.
- Agrega por nivel con `groupby(level=...)`; reorganiza con `swaplevel`, `droplevel`, `rename_axis`.
- Aplana columnas jerárquicas con `"_".join`; vuelve con `str.rsplit(..., expand=True)`.
- Un `DatetimeIndex` ordenado permite `.loc["2018-03"]` y rangos por string.

**Siguiente:** Módulo 03 · `groupby` a fondo: `agg`, `transform`, `filter`, `apply` y agregaciones con nombre.